# Predict Customer Churn — RealMLP (TensorFlow)

RealMLP-style MLP using **TensorFlow/Keras** for `playground-series-s6e3`.  
TensorFlow works natively on Kaggle P100 — no PyTorch cu118 workaround needed.

Pipeline:
1. Imports & GPU setup
2. Load & preprocess data
3. Feature engineering
4. Optuna hyperparameter tuning (50 trials × 5-fold CV)
5. Multi-seed ensemble (5 seeds × 10 folds = 50 models)
6. Save submission + OOF

In [ ]:
# ── Cell 1: Install extras ────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'tqdm'])
print('Done.')

In [ ]:
# ── Cell 2: Imports & GPU setup ───────────────────────────────────────────────
import os, gc, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import optuna
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# GPU memory growth — prevents TF from grabbing all VRAM
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

IS_KAGGLE = os.path.exists('/kaggle/input')
DATA_DIR  = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'
DEVICE    = 'GPU' if gpus else 'CPU'

print(f'Environment : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Device      : {DEVICE} ({len(gpus)} GPU(s))')
if gpus:
    print(f'GPU         : {tf.config.experimental.get_device_details(gpus[0]).get("device_name", "unknown")}')
print(f'TF version  : {tf.__version__}')
print(f'Data dir    : {DATA_DIR}')

In [ ]:
# ── Cell 3: Experiment settings ───────────────────────────────────────────────
RUN_TUNING   = True    # False → skip Optuna, use PRECOMPUTED_PARAMS
N_TRIALS     = 50
N_CV_FOLDS   = 5
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10
TOTAL_MODELS = len(SEEDS) * N_SPLITS
MAX_EPOCHS   = 200
PATIENCE     = 20

PRECOMPUTED_PARAMS = {
    'n_blocks'    : 3,
    'd_block'     : 256,
    'dropout'     : 0.15,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,
    'batch_size'  : 512,
}

print(f'Models : {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS}')
print(f'Tuning : {"ON" if RUN_TUNING else "OFF"} ({N_TRIALS} trials × {N_CV_FOLDS}-fold)')

## 1. Load & Preprocess Data

In [ ]:
print(f'Loading data from {DATA_DIR} ...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
print(f'Train : {train_df.shape}   Test : {test_df.shape}')

TARGET = 'Churn'
if train_df[TARGET].dtype == 'object':
    train_df[TARGET] = train_df[TARGET].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# Joint label encoding
train_df['is_train'] = 1
test_df['is_train']  = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', TARGET, 'is_train']]
categorical_features = []

for col in features:
    if df[col].dtype == 'object':
        categorical_features.append(col)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].fillna('Missing').astype(str))

num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    df[col] = df[col].fillna(df[col].median())

train_enc = df[df['is_train'] == 1].reset_index(drop=True)
test_enc  = df[df['is_train'] == 0].reset_index(drop=True)

print(f'Categoricals : {categorical_features}')
print(f'Numerics     : {num_features}')

## 2. Feature Engineering

In [ ]:
def engineer_features(df):
    """Row-level stats + domain interactions on the three numeric columns."""
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)
    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']  = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio'] = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)
    return df

X      = engineer_features(train_enc[features])
X_test = engineer_features(test_enc[features])
y      = train_enc[TARGET]

X      = pd.get_dummies(X,      columns=categorical_features, dtype=float)
X_test = pd.get_dummies(X_test, columns=categorical_features, dtype=float)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

D_IN = X.shape[1]
print(f'Train : {X.shape}  |  Test : {X_test.shape}  |  d_in = {D_IN}')
print(f'Target dist : {y.value_counts().to_dict()}')

## 3. Model & Training Utilities

In [ ]:
X_np      = X.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)
y_np      = y.values.astype(np.float32)


class RealMLPBlock(layers.Layer):
    """LayerNorm → Dense(ReLU) → Dropout — mirrors RealMLP's residual-free block."""
    def __init__(self, units, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.ln      = layers.LayerNormalization()
        self.dense   = layers.Dense(units, activation='relu')
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        x = self.ln(inputs)
        x = self.dense(x)
        return self.dropout(x, training=training)


def build_model(d_in, params):
    """Build a RealMLP-style Keras model."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(d_in,)))
    for _ in range(params['n_blocks']):
        model.add(RealMLPBlock(params['d_block'], params['dropout']))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=params['lr'],
            weight_decay=params['weight_decay'],
        ),
        loss='binary_crossentropy',
        metrics=['AUC'],
    )
    return model


def train_one_fold(
    X_tr, y_tr, X_val, y_val, X_tst,
    params,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
):
    """Train one fold. Returns (val_probs, test_probs, best_val_auc)."""
    scaler = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_tst_s = scaler.transform(X_tst)

    model = build_model(X_tr_s.shape[1], params)

    es = callbacks.EarlyStopping(
        monitor='val_auc', mode='max',
        patience=patience, restore_best_weights=True,
    )
    lr_sched = callbacks.ReduceLROnPlateau(
        monitor='val_auc', mode='max',
        factor=0.5, patience=patience // 2, min_lr=1e-6,
    )

    model.fit(
        X_tr_s, y_tr,
        validation_data=(X_val_s, y_val),
        epochs=max_epochs,
        batch_size=params['batch_size'],
        callbacks=[es, lr_sched],
        verbose=0,
    )

    val_probs  = model.predict(X_val_s, verbose=0).ravel()
    test_probs = model.predict(X_tst_s, verbose=0).ravel()
    best_auc   = roc_auc_score(y_val, val_probs)

    del model
    keras.backend.clear_session()
    gc.collect()

    return val_probs, test_probs, best_auc


print('train_one_fold() ready.')

## 4. Optuna Hyperparameter Tuning

Set `RUN_TUNING = False` in cell 3 to skip and use `PRECOMPUTED_PARAMS`.

In [ ]:
if RUN_TUNING:
    print(f'Optuna: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV  |  device={DEVICE}')

    pbar = tqdm(total=N_TRIALS, desc='Optuna', unit='trial')
    best_so_far = [0.0]

    def objective(trial):
        params = {
            'n_blocks'    : trial.suggest_int('n_blocks', 1, 5),
            'd_block'     : trial.suggest_categorical('d_block', [64, 128, 256, 384, 512]),
            'dropout'     : trial.suggest_float('dropout', 0.0, 0.5),
            'lr'          : trial.suggest_float('lr', 1e-5, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True),
            'batch_size'  : trial.suggest_categorical('batch_size', [256, 512, 1024, 2048]),
        }

        skf  = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
        aucs = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np)):
            _, _, auc = train_one_fold(
                X_np[tr_idx], y_np[tr_idx],
                X_np[val_idx], y_np[val_idx],
                X_test_np, params,
                max_epochs=100, patience=10,
            )
            aucs.append(auc)
            trial.report(np.mean(aucs), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        score = np.mean(aucs)
        if score > best_so_far[0]:
            best_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_params = study.best_trial.params
    print(f'\nBest CV AUC : {study.best_value:.5f}')
    for k, v in best_params.items():
        print(f'  {k:20s}: {v}')

    study.trials_dataframe().to_csv('realmlp_tf_tuning_results.csv', index=False)
    print('Tuning results saved to realmlp_tf_tuning_results.csv')

else:
    best_params = PRECOMPUTED_PARAMS
    print('Using PRECOMPUTED_PARAMS:')
    for k, v in best_params.items():
        print(f'  {k:20s}: {v}')

## 5. Multi-Seed Ensemble

5 seeds × 10 folds = **50 models** averaged for stable predictions.

In [ ]:
print(f'Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models  |  device={DEVICE}\n')

test_preds_sum = np.zeros(len(X_test_np))
oof_sum        = np.zeros(len(X_np))
fold_auc_log   = []

for seed in tqdm(SEEDS, desc='Seeds'):
    np.random.seed(seed)
    tf.random.set_seed(seed)

    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X_np))

    for fold, (tr_idx, val_idx) in enumerate(
        tqdm(skf.split(X_np, y_np), total=N_SPLITS, desc=f'  Seed {seed}', leave=False)
    ):
        val_preds, test_preds, fold_auc = train_one_fold(
            X_np[tr_idx], y_np[tr_idx],
            X_np[val_idx], y_np[val_idx],
            X_test_np, best_params,
        )
        fold_auc_log.append(fold_auc)
        seed_oof[val_idx] = val_preds
        oof_sum[val_idx] += val_preds
        test_preds_sum   += test_preds

    seed_auc = roc_auc_score(y_np, seed_oof)
    print(f'Seed {seed}  OOF AUC: {seed_auc:.5f}')

global_oof = oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y_np, global_oof)

print(f'\n{"="*55}')
print(f'Fold AUC  mean={np.mean(fold_auc_log):.5f}  std={np.std(fold_auc_log):.5f}')
print(f'Global Ensemble OOF AUC : {final_auc:.5f}')
print(f'{"="*55}')

## 6. Save Submission & OOF

In [ ]:
final_test_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET]      = final_test_preds
sub_file         = 'submission_realmlp_tf.csv'
sub.to_csv(sub_file, index=False)
print(f'Submission  : {sub_file}')
print(f'Pred range  : [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]')

oof_df = pd.DataFrame({'id': train_df['id'], 'oof_pred': global_oof})
oof_df.to_csv('oof_realmlp_tf.csv', index=False)
print(f'OOF         : oof_realmlp_tf.csv  (AUC={final_auc:.5f})')

sub.head()

## 7. Submit to Kaggle

In [ ]:
import subprocess

COMPETITION = 'playground-series-s6e3'
SUBMIT_FILE = sub_file
SUBMIT_MSG  = f'RealMLP TF {TOTAL_MODELS}-model ensemble, Optuna tuned, FE  |  OOF AUC={final_auc:.5f}'
AUTO_SUBMIT = False

print(f'File    : {SUBMIT_FILE}')
print(f'Message : {SUBMIT_MSG}')

if AUTO_SUBMIT:
    result = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', COMPETITION, '-f', SUBMIT_FILE, '-m', SUBMIT_MSG],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
else:
    print(f'\nRun manually:')
    print(f'  kaggle competitions submit -c {COMPETITION} -f {SUBMIT_FILE} -m "{SUBMIT_MSG}"')

### To blend with XGBoost / LightGBM
Add to `notebooks/08_blend_submissions.ipynb`:
```python
{'file': 'submission_realmlp_tf.csv', 'label': 'RealMLP-TF', 'public_score': 0.0},
```